# Домашнее задание 6

`Контекст:` Представьте, что вы работаете в отделе автоматизации почтовых отправлений. Ваша задача — создать систему, которая безошибочно распознает индексы на конвертах. Мы будем использовать классический датасет [MNIST](https://ru.wikipedia.org/wiki/MNIST_(база_данных)) (70,000 изображений рукописных цифр. Каждое 28х28 пикселей) и алгоритм Random Forest.

Ваш путь пройдет от "грубого" обучения до тонкой настройки и визуализации "мыслей". За каждый этап — `1 балл`. Максимум — `5 баллов`.

![alt](../data/13.png)




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA

# Загрузка данных (может занять до 60 секунд времени)
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Задание 1. (1 балл)
Наш архив огромен, а время поджимает. Обычное последовательное обучение заставляет процессор "скучать", пока работает только одно ядро.

**Ваша задача**: Загрузите MNIST и обучите базовый RandomForestClassifier. Но есть нюанс: найдите в документации параметр, который заставляет алгоритм использовать все доступные ядра вашего процессора одновременно.

**Эксперимент**: Замерьте время обучения с этим параметром и без него (на выборке в 10,000 строк). Укажите в ячейке для ответа во сколько раз сократилось время ожидания на вашем устройстве (укажите характеристики вашего устройства: процессор, архитектура, кол-во ядер).

In [ ]:
# Узнайте, перед тем как работать, на чём вы работаете..

import platform
import os

print(f"Процессор: {platform.processor()}")
print(f"Архитектура: {platform.architecture()[0]}")
print(f"Количество ядер: {os.cpu_count()}")

In [ ]:
%%time
# %%timeit - это Cell Magic. Оно измеряет время выполнения всего блока кода в ячейке
# %%time делает то же самое, однако запускает код не 7 раз (для более точного подсчёта), а 1 раз
# Обратите внимание! Код в данной ячейке может выполняться достаточно долго


# Обучите RF в однопоточном формате. Пусть в RF будет 100 деревьев.
rf_single = RandomForestClassifier(n_estimators=100, ...=..., random_state=42)
rf_single.fit(X_train, y_train)

In [ ]:
%%time

# Теперь сделайте замер времени на ВСЕХ ядрах
rf_base = RandomForestClassifier(n_estimators=100, ...=-..., random_state=42)
rf_base.fit(X_train, y_train)


In [ ]:
# Характеристики модели
base_acc = accuracy_score(y_test, rf_base.predict(X_test))
print(f"Точность (Base): {base_acc:.4f}")

## Место для ответа:
1. Название параметра: 
2. Во сколько раз сократилось время работы: 

P. S. Вы можете попробовать реализовать ту же самую задачу при помощи технологии `NVIDIA CUDA` при помощи библиотеки `cuml`. Код сработает в 10-100 раз быстрее, чем `sklearn` на CPU

# Задание 2. (1 балл)
Стандартный лес неплох, но мы хотим создать "идеальный" инструмент. Параметры по умолчанию — это лишь средний вариант.

**Ваша задача:** Используйте `GridSearchCV`. Вам нужно проверить, что важнее: количество деревьев (`n_estimators`) или их глубина (`max_depth`).

**Условие:** После поиска постройте Heatmap (тепловую карту), где по осям будут ваши параметры, а цветом показана точность.

**Вопрос на 1 балл:** Какая комбинация параметров оказалась "золотым стандартом" для наших цифр?


In [ ]:
# Создайте сетку параметров. Переберите n_estimators в значениях 5, 10, 15, 25, 50, 100 и max_depth в 10, 20, None
# param_grid = {}

# Воспользуйтесь GridSearchCV, укажите cv=3, при инициализации RF в качестве estimator укажите многоядерный способ обучения

grid_search = GridSearchCV(
    RandomForestClassifier(n_jobs=-1, random_state=42),
    param_grid, cv=3
)
grid_search.fit(X_train[:5000], y_train[:5000])

# Визуализация Heatmap
results = pd.DataFrame(grid_search.cv_results_)
pivot = results.pivot_table(index='param_max_depth', columns='param_n_estimators', values='mean_test_score')

plt.figure(figsize=(7, 5))
sns.heatmap(pivot, annot=True, cmap='hit', fmt='.3f')
plt.title("GridSearch: Влияние глубины и кол-ва деревьев")
plt.show()


# С помощью .best_estimator_ достаньте .best_params_
best_rf = ...

## Место для ответа:
Какие параметры являются наилучшими и с какими значениями:

1.

2. 

Почему так? Напишите своё мнение, нам интересно :)

# Задание 3. (1 балл)

Заказчик спрашивает: "А как ваш алгоритм понимает, что это именно восемь, а не ноль?". Пришло время визуализировать "внимание" модели.

**Ваша задача:** Обучите 10 отдельных лесов (для каждой цифры свой бинарный классификатор: "это 5" или "это не 5"). Извлеките `feature_importances_` для каждой цифры.

**Визуализация:** Выведите сетку 2х5, где для каждой цифры показана карта важности пикселей (используйте `cmap='hot'`).

**Вопрос на 1 балл:** Какие ключевые зоны (центр, края, петли) выделяет модель?


In [ ]:
# Запустите исходный код
# Поменяйте параметр n_estimators=10, но оставьте max_depth = 1
# Теперь наоборот: n_estimators=1, а max_depth = 10
# Теперь попробуйте:n_estimators=15 и max_depth = 15 (кажется, что вы начинаете видеть то, что видит RF)
# Подберите лучшие n_estimators и max_depth для визуализации результата

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
fig.suptitle('Карты значимости пикселей', fontsize=16)

for digit in range(10):
    y_bin = (y_train == str(digit)).astype(int)
    rf_bin = RandomForestClassifier(n_estimators=1, max_depth=1, ...=..., random_state=42)
    rf_bin.fit(X_train[:20000], y_bin[:20000]) # Берем часть данных для скорости

    importances = rf_bin.feature_importances_.reshape(28, 28)

    ax = axes[digit // 5, digit % 5]
    # Используйте гауссово размытие для красоты
    ax.imshow(importances, cmap=..., interpolation=...)
    ax.axis('off') # Отключаем оси для красоты
    ax.set_title(f"Важность для '{digit}'")

plt.tight_layout()
plt.show()

## Место для ответа:
Какой распределение пикселей характено для 0?
Какое распределение пикселей действительно похоже на распозноваемый объект, по вашему мнению?


# Задание 4. (1 балл)

Даже лучший детектив иногда ошибается. Нам нужно знать "врага в лицо" — какие цифры модель путает чаще всего?

**Ваша задача:** Постройте матрицу ошибок (`Confusion Matrix`). Найдите в ней самую "проблемную" пару цифр (где число ложных срабатываний максимально).

**Эксперимент:** Найдите в тестовом наборе 5 реальных картинок, где модель перепутала именно эти две цифры. Выведите их на экран.

**Вопрос на 1 балл:** Какую пару цифр модель путает чаще всего? Глядя на картинки ошибок, как вы думаете — почему?

In [ ]:
y_pred = best_rf.predict(X_test)

cm = ... # Пригодится confusion_matrix

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')
plt.title("Матрица ошибок (Confusion Matrix)")
plt.xlabel("Предсказано")
plt.ylabel("Реально")
plt.show()

# Ищем ошибки:
# error_mask = np.where((y_test ==  ...) & (y_pred == ...))[0]

plt.figure(figsize=(10, 3))
for i, idx in enumerate(error_mask[:5]):
    plt.subplot(1, 5, i+1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap='hot')
    plt.title(f"Ошибка №{i+1}")
    plt.axis('off')
plt.show()

## Место для ответа:
Какую пару цифр модель путает чаще всего? Глядя на картинки ошибок, как вы думаете — почему?


# Задание 5. (1 балл)
Финальный босс. Нам нужно запустить нашу модель на слабом почтовом терминале. Памяти мало, данных много.
**Ваша задача:** Примените метод PCA, чтобы оставить только 95% полезной информации (дисперсии). Посмотрите, до скольких признаков сократился вектор (было 784).

**Финальный тест:** Обучите лес на сжатых данных (используя лучшие параметры из задания №2). Сравните время обучения и точность с заданием №1.

**Вопрос на 1 балл:** Сколько компонент осталось после PCA? На сколько процентов упала точность при таком мощном сжатии? Стоила ли игра свеч?


In [ ]:
pca = ...
X_train_pca = ...
X_test_pca = ...


In [ ]:
%%time
rf_pca = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf_pca.fit(X_train_pca, y_train)

In [ ]:
acc_pca = accuracy_score(y_test, rf_pca.predict(X_test_pca))

print(f"Признаков после PCA: {X_train_pca.shape[1]} (было 784)")
print(f"Accuracy с PCA: {acc_pca:.4f}")
print(f"Ускорение обучения: {time_multi / time_pca:.1f}x")

## Место для ответа:
Сколько компонент осталось после PCA? На сколько процентов упала точность при таком мощном сжатии? Стоила ли игра свеч?
